# Fig6, stage 02: apply the gating strategy

The three polygons in `gates.npz` are applied to every cell.  A cell is kept only
when it is inside all three, which is the sequential gating strategy: a cell, a
single cell, and alive.

## Inputs

| file | produced by | holds |
|---|---|---|
| `Fig6.h5ad` | stage 00 | every cell of every well |
| `gates.npz` | stage 01 of Loop1, copied into this folder | the three gate polygons |

## Outputs

| file | holds |
|---|---|
| `Fig6_Gated.h5ad` | every cell that passed all three gates |
| `Fig6_Gated_500k.h5ad` | about 500,000 of those cells |

Stage 04 reads `Fig6_Gated.h5ad`.

Run the cells from top to bottom.

In [ ]:
import matplotlib.pyplot as plt
import pytometry as pm
import scanpy as sc

from labcompass import apply_saved_gates, downsample, target_size_for

IN_ALL = "Fig6.h5ad"
GATE_FILE = "gates.npz"

OUT_GATED = "Fig6_Gated.h5ad"
OUT_GATED_SUBSET = "Fig6_Gated_500k.h5ad"

DESIRED_CELL_NUMBER = 500000
DOWNSAMPLE_SEED = 0

In [ ]:
adata_all = sc.read(IN_ALL)
print(f"{adata_all.n_obs:,} cells before gating")
adata_all

## Apply the gates

In [ ]:
before = adata_all.n_obs
adata_all, masks = apply_saved_gates(adata_all, GATE_FILE, mode="all")

for i, mask in enumerate(masks, start=1):
    print(f"gate {i} alone keeps {mask.sum():,} of {before:,} cells")
print(f"all three together keep {adata_all.n_obs:,} cells "
      f"({100 * adata_all.n_obs / before:.1f} per cent)")

The live gate is drawn against 7-AAD.  After gating, the bright events should be gone.

In [ ]:
pm.pl.scatter_density(
    adata_all, x="FSC-A :: FSC - Area", y="7-AAD",
    x_lim=[0, 1_500_000], y_lim=[-10_000, 10_000],
)

In [ ]:
plt.close("all")

In [ ]:
adata_all.write(OUT_GATED)
print(f"wrote {OUT_GATED}: {adata_all.n_obs:,} cells")

## Cells kept per experiment

In [ ]:
counts = adata_all.obs["experiment_number"].value_counts()
ax = counts.plot.bar(figsize=(max(6, 0.2 * len(counts)), 4))
ax.set_ylabel("cells kept")
ax.set_xlabel("experiment number")
plt.tight_layout()
plt.show()
counts

## Write the downsampled gated object

In [ ]:
target_size = target_size_for(adata_all, DESIRED_CELL_NUMBER, obs="experiment_number")
print(f"cap of {target_size:,} cells per experiment")
downsample(
    adata_all, obs="experiment_number", target_size=target_size, seed=DOWNSAMPLE_SEED
).write(OUT_GATED_SUBSET)
print(f"wrote {OUT_GATED_SUBSET}")